# 00. Entorno, corpus y dataset

El experimento compara cuatro estrategias de recuperación y tres generadores sobre preguntas con dos formulaciones. Antes de recalcular las métricas, conviene comprobar qué documentos, fragmentos y preguntas forman el conjunto de análisis.

Este cuaderno lee los materiales incluidos. No descarga documentos ni vuelve a extraer los PDF.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

## Entorno y archivos de entrada

Los scripts se ejecutan con el mismo Python que el kernel. Las rutas siguientes pertenecen al paquete de reproducción; no apuntan a otra versión del TFM.

In [2]:
import platform
from importlib.metadata import version

display(pd.DataFrame({'componente': ['Python', 'pandas', 'nbclient', 'PyMuPDF'],
                      'version': [platform.python_version(), version('pandas'),
                                  version('nbclient'), version('pymupdf')]}))
paths = ['config/corpus_sources.csv', 'data/processed/chunks_v1.jsonl',
         'data/golden/variants/generation_golden_v3.jsonl',
         'outputs/local_runs/explicitness_v3/qrels_final_v3.jsonl']
assert all((ROOT / path).is_file() for path in paths)
display(pd.DataFrame({'entrada': paths, 'disponible': True}))

,componente,version
0,Python,3.14.0
1,pandas,3.0.3
2,nbclient,0.11.0
3,PyMuPDF,1.28.2


,entrada,disponible
0,config/corpus_sources.csv,True
1,data/processed/chunks_v1.jsonl,True
2,data/golden/variants/generation_golden_v3.jsonl,True
3,outputs/local_runs/explicitness_v3/qrels_final_v3.jsonl,True


## Documentos del corpus

El corpus procede de dos manuales abiertos de OpenStax. El archivo de fuentes conserva la licencia, la URL y los capítulos seleccionados. Los PDF originales se mantienen como referencia de la extracción.

In [3]:
sources = pd.read_csv(ROOT / 'config/corpus_sources.csv')
assert len(sources) == 2
assert all((ROOT / path).is_file() for path in sources['raw_path'])
display(sources[['title', 'language', 'license', 'included_chapters', 'source_url']])

,title,language,license,included_chapters,source_url
0,Introductory Statistics 2e,en,CC BY-NC-SA 4.0,6-9,https://openstax.org/details/books/introductory-statistics-2e
1,Principles of Data Science,en,CC BY-NC-SA 4.0,3-4,https://openstax.org/details/books/principles-data-science


## Fragmentos utilizados para recuperar evidencia

La segmentación respeta las secciones y utiliza solapamiento. El tamaño se expresa en palabras, no en tokens del generador. El campo formula_risk señala fragmentos cuya extracción puede haber perdido contenido matemático.

In [4]:
chunks = pd.DataFrame(read_jsonl(ROOT / 'data/processed/chunks_v1.jsonl'))
chunking = json.loads((ROOT / 'data/processed/chunking_report_v1.json').read_text())
assert len(chunks) == chunks['chunk_id'].nunique() == chunking['total_chunks'] == 288
assert chunks['doc_id'].nunique() == 2
display(chunks.groupby(['source_title', 'chapter_id', 'chapter_title'], sort=False)
        .agg(fragmentos=('chunk_id', 'size'), palabras_mediana=('word_count', 'median'))
        .reset_index())

,source_title,chapter_id,chapter_title,fragmentos,palabras_mediana
0,Introductory Statistics 2e,6,The Normal Distribution,23,287.0
1,Introductory Statistics 2e,7,The Central Limit Theorem,31,291.0
2,Introductory Statistics 2e,8,Confidence Intervals,43,290.0
3,Introductory Statistics 2e,9,Hypothesis Testing with One Sample,42,291.5
4,Principles of Data Science,3,Descriptive Statistics: Statistical Measurements and Probability Distributions,58,291.5
5,Principles of Data Science,4,Inferential Statistics and Regression Analysis,91,291.0


In [5]:
display(pd.DataFrame([chunking['parameters']]))
display(chunks['word_count'].describe().to_frame('palabras_por_fragmento'))
display(chunks['formula_risk'].value_counts(dropna=False).rename_axis('riesgo_extraccion').to_frame('fragmentos'))

,boundary,target_words,max_words,min_words,overlap_words,overlap_unit
0,section,300,380,100,60,complete_text_units


,palabras_por_fragmento
count,288.000000
mean,281.625000
std,32.880287
min,107.000000
25%,281.000000
50%,291.000000
75%,296.000000
max,327.000000


,fragmentos
riesgo_extraccion,
False,283
True,5


Para comprobar qué llega al recuperador, se puede leer un fragmento completo. Los identificadores conectan este texto con los rankings, las citas y los juicios de relevancia.

In [6]:
example = chunks.loc[chunks['chunk_id'] == 'openstax_intro_stats_2e:7.1:chunk-001'].iloc[0]
print(example['chunk_id'])
print(example['retrieval_text'])

openstax_intro_stats_2e:7.1:chunk-001
Source: Introductory Statistics 2e
Chapter 7: The Central Limit Theorem
Section 7.1: The Central Limit Theorem for Sample Means (Averages)

Each time a person rolls more than one die, they calculate the sample mean of the faces showing. For example, one 
person might roll five fair dice and get 2, 2, 3, 4, 6 on one roll.

The mean is 
 = 3.4. The 3.4 is one mean when five fair dice are rolled. This same person would roll the

five dice nine more times and calculate nine more means for a total of ten means.

Your instructor will pass out the dice to several people. Roll your dice ten times. For each roll, record the faces, and 
find the mean. Round to the nearest 0.5.

Your instructor (and possibly you) will produce one graph (it might be a histogram) for one die, one graph for two 
dice, one graph for five dice, and one graph for ten dice. Since the "mean" when you roll one die is just the face on the 
die, what distribution do these means appear t

## Unidades del experimento final

Una pregunta base tiene dos formulaciones. Cada formulación se combina con cuatro recuperadores y tres generadores; las respuestas de una misma base siguen siendo observaciones relacionadas. Los contrastes posteriores agregan por pregunta base.

In [7]:
golden = pd.DataFrame(read_jsonl(ROOT / 'data/golden/variants/generation_golden_v3.jsonl'))
assert len(golden) == golden['qid'].nunique() == 52
assert golden['base_qid'].nunique() == 26
assert golden.groupby('base_qid').size().eq(2).all()
units = pd.DataFrame({'unidad': ['Preguntas base', 'Formulaciones', 'Recuperadores', 'Generadores', 'Combinaciones previstas'],
                      'n': [golden['base_qid'].nunique(), len(golden), 4, 3, len(golden) * 4 * 3]})
display(units)

,unidad,n
0,Preguntas base,26
1,Formulaciones,52
2,Recuperadores,4
3,Generadores,3
4,Combinaciones previstas,624


El cuaderno 01 examina las preguntas y sus evidencias. Los resultados de recuperación se recalculan en el 02; las respuestas y su evaluación se revisan en el 03.